# Kenya questions: from the students' sheets to the test set

Run this notebook to turn the students' spreadsheets into the Kenya questions the model is tested
on, and **see every step**: which file is which student, every change the clean-up makes, what
gets imported, and the final checks. It runs on the laptop (no GPU) in about a minute.

**How to run:** top right **Select Kernel → `bluff-detector (.venv)`**, then **Run All**.
Re-run it whenever a student sends an updated sheet (after updating `data/kenya/student_files.yaml`).

| Step | What happens | Code |
|---|---|---|
| 1 | Which file is which student | `data/kenya/student_files.yaml` |
| 2 | Clean the sheets and apply the AI repairs; **every change is listed** | `src/bluff/sheet_clean.py`, `data/kenya/repairs.yaml` |
| 3 | Look at the changes | `data/kenya/cleaned/changes.csv` |
| 4 | Import into the question file | `src/bluff/sheet_import.py` |
| 5 | Check: data contract and leakage audit | `src/bluff/pipeline/validate.py`, `audit.py` |
| 6 | The final questions | `data/kenya/questions.jsonl` |

The full explanation: `docs/kenya-questions.md`. What each student still has to fix:
`..\Kenya questions - fixes per student.md`.

In [1]:
import os
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

from bluff import sheet_clean, sheet_import
from bluff.config import load_config
from bluff.pipeline import audit, ingest, validate
from bluff.utils import read_jsonl

if Path.cwd().name == "notebooks":  # work from the project folder
    os.chdir("..")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)
cfg = load_config("configs/variants/m0_base.yaml")

## 1. Which file is which student

From `data/kenya/student_files.yaml`, matched to the Student Brief's letters and domains. Edit
that file (not this notebook) when a student sends a new sheet or a name becomes known. Rows
listed under `exclude` are left out on purpose, with the reason.

In [2]:
specs = yaml.safe_load(Path("data/kenya/student_files.yaml").read_text(encoding="utf-8"))
display(pd.DataFrame([{
    "student": s["student"], "name": s["author"], "domain": s["domain"],
    "file": s["file"], "tab": s["tab"], "file found": Path(s["file"]).exists(),
    "rows excluded": len(s.get("exclude") or {}),
} for s in specs]).set_index("student"))

,name,domain,file,tab,file found,rows excluded
student,,,,,,
A,James Gosha,civics_law,../Bluff Detector - Kenya Question Sheet student b.xlsx,Student B,True,0
B,Student B,geography_society,../Geography_Questions.xlsx,Sheet1,True,0
C,Michael,institutions_services,../Bluff Detector - Kenya Question Sheet 1.xlsx,Student C,True,0
D,Student D,western_default_traps,../project 2.xlsx,Student D,True,0


## 2. Clean the sheets

Only mechanical fixes that don't change what a question asks: add a missing "?", add
"In Kenya, " when the question never names the country, show Excel-damaged answers as typed
(16% stored as 0.16, dates), map the domain onto the brief's six, split "999/112" answers, fill a
missing Author, and skip rows that can't be used (empty, headings, no answer, about another
country, repeats). **Then the AI review's repairs** in `data/kenya/repairs.yaml` (19 November):
rows dropped or edited with a reason, UK twins and replacement fakes written; every new row has
Author "Claude (AI)" and every edited row Reviewer "Claude (AI)" (see `AI_USE.md`). Writes
`data/kenya/cleaned/cleaned_questions.xlsx` and the change list.

In [3]:
result = sheet_clean.run(cfg)
counts = pd.DataFrame(result["students"]).fillna(0).astype(int)
print(f"{result['changes']} changes recorded")
display(counts)

599 changes recorded


,Student A (James Gosha),Student B (Student B),Student C (Michael),Student D (Student D)
"added 'In Kenya, '",16,11,6,1
domain mapped to the brief's six,46,71,52,51
kept: real,71,60,32,41
"AI review, edited",8,19,7,87
"AI review, dropped",3,13,34,31
Excel value shown as typed,9,2,1,1
kept: fake,20,20,20,10
skipped: no question text,15,0,0,1
AI-written UK twin,16,8,5,4
Author filled from student_files.yaml,0,80,0,51


## 3. Every change, row by row

The same list is in the **Changes** tab of `cleaned_questions.xlsx` and in `changes.csv`. Below:
how many of each kind, then a few examples of each. To see all of one kind, change `KIND`.

In [4]:
changes = pd.read_csv("data/kenya/cleaned/changes.csv", encoding="utf-8-sig")
display(changes.groupby(["why", "student"]).size().unstack(fill_value=0))

student,Student A,Student B,Student C,Student D
why,,,,
"AI review, dropped: 'major' is ambiguous; Student D asks the same fact precisely",0,1,0,0
"AI review, dropped: a changing fact with no 'Valid until' (rule 4)",0,0,0,4
"AI review, dropped: a list answer",0,0,1,0
"AI review, dropped: a list answer can't be marked right or wrong (rule 1)",0,1,0,0
"AI review, dropped: a reserved example from the Student Brief",2,0,1,2
...,...,...,...,...
"skipped: about another country, not Kenya",0,0,0,10
skipped: no Type (a heading or unfinished row),0,8,0,1
skipped: no answer (a real or twin question),0,1,1,0


In [5]:
for why, g in changes.groupby("why"):
    display(Markdown(f"**{why}** ({len(g)})"))
    display(g[["student", "row", "before", "after"]].head(3))

**AI review, dropped: 'major' is ambiguous; Student D asks the same fact precisely** (1)

,student,row,before,after
126,Student B,17,How many major cities are there in Kenya?,NaN


**AI review, dropped: a changing fact with no 'Valid until' (rule 4)** (4)

,student,row,before,after
383,Student D,3,what is the Kenya's total population,NaN
384,Student D,4,what is Kenya's total GDP,NaN
443,Student D,29,how many banks does kenya has,NaN


**AI review, dropped: a list answer** (1)

,student,row,before,after
306,Student C,36,What activities does NTSA conduct on motor vehicles?,NaN


... (170 outputs left out)

**skipped: no question text** (16)

,student,row,before,after
82,Student A,96,twin,NaN
83,Student A,97,twin,NaN
84,Student A,98,twin,NaN


**skipped: repeats an earlier question or a starter** (4)

,student,row,before,after
123,Student B,14,What is the capital city of Kenya?,NaN
317,Student C,42,Which department issues Kenyan passports?,NaN
334,Student C,56,Which agency certifies horticultural exports for phytosanitary compliance?,NaN


In [6]:
KIND = "added 'In Kenya, '"  # any value from the 'why' column above
changes[changes["why"] == KIND][["student", "row", "before", "after"]]

,student,row,before,after
0,Student A,2,Who is incharge of the registration of candidates for election?,"In Kenya, who is incharge of the registration of candidates for election?"
2,Student A,3,Who is in charge of the settlement of electoral disputes?,"In Kenya, who is in charge of the settlement of electoral disputes?"
4,Student A,4,Who is in charge of voter education?,"In Kenya, who is in charge of voter education?"
6,Student A,5,Who is in charge of the delimitation of constituencies and wards?,"In Kenya, who is in charge of the delimitation of constituencies and wards?"
8,Student A,6,"Who is in charge of facilitating the observation, monitoring and evaluation ...","In Kenya, who is in charge of facilitating the observation, monitoring and e..."
12,Student A,8,Who is in charge of the continuous registration of voters and the revision o...,"In Kenya, who is in charge of the continuous registration of voters and the ..."
14,Student A,9,Who is in charge of monitoring compliance with the legislation on the nomina...,"In Kenya, who is in charge of monitoring compliance with the legislation on ..."
16,Student A,10,who is the administrative head of the national assembly?,"In Kenya, who is the administrative head of the national assembly?"
18,Student A,11,who is incharge of management of parliamentary internship programmes?,"In Kenya, who is incharge of management of parliamentary internship programmes?"
20,Student A,12,who maintains custody of the mace in parliament?,"In Kenya, who maintains custody of the mace in parliament?"


## 4. Import into the question file

Each cleaned row becomes one question in `data/kenya/questions.jsonl` (real → `kenya`, fake →
`kenya_unanswerable`, twin → `western`), with an id made from its text, so re-importing keeps
every earlier id and the GPU run only asks new or changed questions. Trevor's 9 starter
questions are kept.

In [7]:
imported = sheet_import.run(cfg, "data/kenya/cleaned/cleaned_questions.xlsx")
print(f"{imported['imported']} imported from the students, {imported['starters_kept']} starters "
      f"kept: {imported['total']} Kenya questions (was {imported['previous_total']})")
display(pd.DataFrame(imported["by_author"]).T)
display(Markdown(Path(imported["report"]).read_text(encoding="utf-8")))

317 imported from the students, 9 starters kept: 326 Kenya questions (was 326)


,kenya,kenya_unanswerable,western,verified
Trevor,7,2,0,0
JAMES GOSHA,71,20,0,0
Student B,60,20,0,0
Michael,32,20,0,0
Student D,41,10,0,0
Claude (AI),0,10,33,0


# Kenya question import report

Imported 2026-11-19 23:30 from `cleaned_questions.xlsx`: 317 filled rows; the question file now holds 326 questions (it held 326).

## Questions per author

| Author | Real | Fake | Twin | Verified |
|---|---|---|---|---|
| Claude (AI) | 0 | 10 | 33 | 0 |
| JAMES GOSHA | 71 | 20 | 0 | 0 |
| Michael | 32 | 20 | 0 | 0 |
| Student B | 60 | 20 | 0 | 0 |
| Student D | 41 | 10 | 0 | 0 |
| Trevor | 7 | 2 | 0 | 0 |

## Rows with problems (0)

None.


## 5. Checks

**Data contract** (`validate`): ids unique, real questions have answers, fakes have none, pools
correct. **Leakage audit** (`audit`): no question's wording may appear in the documents the RAG
model reads or the fine-tuned model is trained on; a question that copies a source would make
the test unfair. The audit needs the full question list, so `ingest` runs first (it uses the
cached TriviaQA download).

In [8]:
print("data contract:", validate.run(cfg))
m1 = load_config("configs/variants/m1_rag.yaml")
ingest.run(m1)
try:
    print("leakage audit:", audit.run(m1))
except ValueError as e:
    print("LEAKAGE AUDIT FAILED:", e)
    print("See data/processed/leakage_audit.json; exclude the rows in student_files.yaml.")

326 of 326 Kenya items are not yet verified by a second reviewer


data contract: {'n': 326, 'answerable': 244, 'unanswerable': 82, 'verified': 0}


C:\Users\Admin\OneDrive\Desktop\Bluff_Detector\bluff-detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


leakage audit: {'retrieval_corpus': {'checked': True, 'overlaps': 0, 'examples': []}, 'finetune_corpus': {'checked': True, 'overlaps': 0, 'examples': []}}


## 6. The final Kenya questions

What the model will be asked, per student and type, and a few examples of each. Next: upload to
GitHub (`git add data/kenya`, `git commit`, `git push`), then run `notebooks/run_m0_base.ipynb`
on Kaggle or Colab: it asks only the new questions.

In [9]:
q = pd.DataFrame(read_jsonl(cfg["data"]["kenya_questions"]))
display(pd.crosstab(q["author"], q["pool"], margins=True, margins_name="total"))
print("Written by the AI review:", int((q["author"] == "Claude (AI)").sum()),
      "| edited by it:", int((q["reviewer"] == "Claude (AI)").sum()))
examples = pd.concat([d.sample(min(4, len(d)), random_state=1) for _, d in q.groupby("pool")])
display(examples[["id", "pool", "author", "question", "gold"]])

pool,kenya,kenya_unanswerable,western,total
author,,,,
Claude (AI),0,10,33,43
JAMES GOSHA,71,20,0,91
Michael,32,20,0,52
Student B,60,20,0,80
Student D,41,10,0,51
Trevor,7,2,0,9
total,211,82,33,326


Written by the AI review: 43 | edited by it: 56


,id,pool,author,question,gold
46,ke-sw-4e19fa5e,kenya,JAMES GOSHA,Siku ya Mashujaa huadhimishwa tarehe ngapi?,20 Oktoba
251,ke-trap-7795acb7,kenya,Student D,Which service is responsible for criminal investigations in Kenya?,DCI
53,ke-ins-c59cae30,kenya,JAMES GOSHA,Which commission investigates corruption in Kenya?,Ethics and Anti-Corruption Commission
261,ke-trap-64fef809,kenya,Student D,What is Kenya's largest category of imports by value?,Petroleum products
88,ke-una-civ-a40dcb88,kenya_unanswerable,JAMES GOSHA,Which Kenyan county borders the Atlantic Ocean?,NaN
281,ke-una-trap-0da60157,kenya_unanswerable,Student D,who discovered the law of motion in Kenya?,NaN
217,ke-una-ins-9932cf8d,kenya_unanswerable,Michael,Was KRA established on 1st July 9998?,NaN
224,ke-una-ins-a19cde97,kenya_unanswerable,Michael,Is NTSA the State Corporation in Kenya that selects students for admission t...,NaN
307,uk-civ-9b3da4ef,western,Claude (AI),What is the standard VAT rate in the United Kingdom?,20%
312,uk-geo-5a1111bc,western,Claude (AI),What is the longest river in the United Kingdom?,River Severn
